# BiSON Performance — Implementation / 구현

**Paper**: Chaplin, W.J., Elsworth, Y., Howe, R., Isaak, G.R., McLeod, C.P., Miller, B.A., van der Raay, H.B., Wheeler, S.J., New, R. (1996). "BiSON Performance." *Solar Physics*, 168, 1–18.

이 노트북은 BiSON 논문의 핵심 개념을 구현합니다:
This notebook implements the key concepts from the BiSON paper:

1. **Part 1**: 공명 산란 분광기 시뮬레이션 / Resonant scattering spectrometer simulation
2. **Part 2**: 네트워크 duty cycle 모델링 / Network duty cycle modeling
3. **Part 3**: Window function과 sideband 분석 / Window function and sideband analysis
4. **Part 4**: 저차수 p-모드 파워 스펙트럼 생성 / Low-degree p-mode power spectrum generation
5. **Part 5**: 논문 Table II–IV 성능 데이터 재현 / Reproducing paper Tables II–IV performance data
6. **Part 6**: 관측소 수와 duty cycle 관계 / Station count vs. duty cycle relationship

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from typing import NamedTuple

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Seed for reproducibility
RNG = np.random.default_rng(42)

## Part 1: Resonant Scattering Spectrometer Simulation / 공명 산란 분광기 시뮬레이션

BiSON의 핵심 측정 원리를 시뮬레이션합니다. 칼륨(K I 770 nm) Fraunhofer 흡수선의 도플러 이동을 공명 산란 비 $\mathcal{R}$로 측정합니다:

We simulate the core measurement principle of BiSON. The Doppler shift of the potassium (K I 770 nm) Fraunhofer absorption line is measured via the scattered ratio $\mathcal{R}$:

$$\mathcal{R} = \frac{I_B - I_R}{I_B + I_R}, \quad V_{\text{obs}} = k\mathcal{R}, \quad k \approx 3000\,\text{m\,s}^{-1}$$

광자 통계에 의한 소음 한계 / Photon-statistics noise floor:

$$\mathrm{d}V_{\text{obs}} \approx \frac{k}{\sqrt{I_B + I_R}} \approx 1.5\,\text{cm\,s}^{-1} \quad (\text{40-s integration})$$

In [ ]:
def simulate_k_line_profile(
    wavelength: np.ndarray, line_center: float = 770.0, depth: float = 0.6,
    width: float = 0.02
) -> np.ndarray:
    """Simulate a Gaussian absorption line profile for K I 770 nm.

    Args:
        wavelength: Wavelength array in nm.
        line_center: Central wavelength of the absorption line in nm.
        depth: Fractional depth of the absorption line (0 to 1).
        width: Gaussian sigma width in nm.

    Returns:
        Normalized intensity profile.
    """
    return 1.0 - depth * np.exp(-0.5 * ((wavelength - line_center) / width) ** 2)


def scattered_ratio(
    profile: np.ndarray, wavelength: np.ndarray, line_center: float,
    wing_offset: float = 0.015
) -> float:
    """Compute the resonant scattered ratio R = (I_B - I_R) / (I_B + I_R).

    Args:
        profile: Intensity profile array.
        wavelength: Wavelength array in nm.
        line_center: Central wavelength in nm.
        wing_offset: Offset from line center to blue/red wing sampling points.

    Returns:
        Scattered ratio R.
    """
    idx_blue = np.argmin(np.abs(wavelength - (line_center - wing_offset)))
    idx_red = np.argmin(np.abs(wavelength - (line_center + wing_offset)))
    i_b = profile[idx_blue]
    i_r = profile[idx_red]
    return (i_b - i_r) / (i_b + i_r)


# Simulate the spectrometer response to different velocities
K_LINE_CENTER = 770.0  # nm
K_PROPORTIONALITY = 3000.0  # m/s
C_LIGHT = 3e8  # m/s

wavelength = np.linspace(769.90, 770.10, 2000)
velocities = np.linspace(-500, 500, 200)  # m/s

ratios = []
for v in velocities:
    # Doppler shift: delta_lambda / lambda = v / c
    shifted_center = K_LINE_CENTER * (1 + v / C_LIGHT)
    profile = simulate_k_line_profile(wavelength, line_center=shifted_center)
    r = scattered_ratio(profile, wavelength, K_LINE_CENTER)
    ratios.append(r)

ratios = np.array(ratios)
v_measured = K_PROPORTIONALITY * ratios

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Absorption line profile at rest and shifted
ax = axes[0]
profile_rest = simulate_k_line_profile(wavelength, line_center=K_LINE_CENTER)
shifted = K_LINE_CENTER * (1 + 200 / C_LIGHT)
profile_shifted = simulate_k_line_profile(wavelength, line_center=shifted)
ax.plot(wavelength, profile_rest, "b-", label="At rest / 정지")
ax.plot(wavelength, profile_shifted, "r--", label=f"v = +200 m/s")
ax.axvline(K_LINE_CENTER - 0.015, color="blue", alpha=0.3, ls=":", label="Blue wing")
ax.axvline(K_LINE_CENTER + 0.015, color="red", alpha=0.3, ls=":", label="Red wing")
ax.set_xlabel("Wavelength (nm) / 파장")
ax.set_ylabel("Normalized Intensity / 정규화 강도")
ax.set_title("(a) K I 770 nm Absorption Line / 흡수선")
ax.legend(fontsize=9)

# (b) Scattered ratio vs velocity — linearity check
ax = axes[1]
ax.plot(velocities, ratios, "k-", lw=2)
ax.set_xlabel("True velocity (m/s) / 실제 속도")
ax.set_ylabel(r"Scattered ratio $\mathcal{R}$")
ax.set_title(r"(b) $\mathcal{R}$ vs Velocity / 속도 대 산란 비")

# Linear fit
coeffs = np.polyfit(velocities, ratios, 1)
ax.plot(velocities, np.polyval(coeffs, velocities), "r--", alpha=0.7,
        label=f"Linear fit: slope = {coeffs[0]:.2e}")
ax.legend(fontsize=9)

# (c) Measured velocity vs true velocity
ax = axes[2]
ax.plot(velocities, v_measured, "k-", lw=2)
ax.plot(velocities, velocities, "r--", alpha=0.7, label="1:1 line")
ax.set_xlabel("True velocity (m/s) / 실제 속도")
ax.set_ylabel("Measured velocity (m/s) / 측정 속도")
ax.set_title(f"(c) $V_{{obs}} = k\\mathcal{{R}}$, k = {K_PROPORTIONALITY} m/s")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Linearity: R² = {np.corrcoef(velocities, ratios)[0,1]**2:.8f}")
print(f"Effective k from fit: {1/coeffs[0]:.1f} m/s (paper value: 3000 m/s)")

## Part 2: Network Duty Cycle Modeling / 네트워크 Duty Cycle 모델링

BiSON 6개 관측소의 위치(Table I)를 기반으로, 각 관측소의 일조 시간(sunlight hours)을 천문학적으로 계산하고, 네트워크의 이론적 duty cycle을 시뮬레이션합니다.

Based on the 6 BiSON station locations (Table I), we astronomically compute sunlight hours at each station and simulate the network's theoretical duty cycle.

관측소가 태양을 관측할 수 있는 조건: 태양 고도(altitude) > 0° (실제로는 대기 영향으로 ~10° 이상이 필요하지만, 여기서는 단순화).

Condition for observing: solar altitude > 0° (in practice >~10° needed due to atmospheric effects, simplified here).

In [ ]:
class Station(NamedTuple):
    """A BiSON observing station."""
    name: str
    longitude: float  # degrees East (West is negative)
    latitude: float   # degrees North (South is negative)


# BiSON stations from Table I (p. 4)
BISON_STATIONS = [
    Station("Mount Wilson", -118.067, 34.133),
    Station("Las Campanas", -70.700, -29.017),
    Station("Izaña", -16.500, 28.300),
    Station("Sutherland", 20.817, -32.383),
    Station("Carnarvon", 113.750, -24.850),
    Station("Narrabri", 149.567, -30.317),
]


def solar_declination(day_of_year: np.ndarray) -> np.ndarray:
    """Compute solar declination angle for a given day of year.

    Args:
        day_of_year: Day of year (1-365).

    Returns:
        Solar declination in degrees.
    """
    return 23.44 * np.sin(np.radians((284 + day_of_year) * 360 / 365))


def hour_angle_sunset(latitude: float, declination: np.ndarray) -> np.ndarray:
    """Compute the hour angle of sunset for given latitude and declination.

    Args:
        latitude: Observer latitude in degrees.
        declination: Solar declination in degrees.

    Returns:
        Hour angle of sunset in degrees. NaN if sun never rises/sets.
    """
    lat_rad = np.radians(latitude)
    dec_rad = np.radians(declination)
    cos_ha = -np.tan(lat_rad) * np.tan(dec_rad)
    cos_ha = np.clip(cos_ha, -1, 1)
    return np.degrees(np.arccos(cos_ha))


def sunlight_hours(latitude: float, day_of_year: np.ndarray) -> np.ndarray:
    """Compute hours of sunlight for a station on given days.

    Args:
        latitude: Station latitude in degrees.
        day_of_year: Array of day-of-year values.

    Returns:
        Hours of sunlight per day.
    """
    dec = solar_declination(day_of_year)
    ha = hour_angle_sunset(latitude, dec)
    return 2.0 * ha / 15.0  # Convert degrees to hours (15 deg/hr)


def is_sun_up(
    longitude: float, latitude: float, day_of_year: float, ut_hour: float,
    min_altitude: float = 10.0
) -> bool:
    """Check if the Sun is above minimum altitude at a given time and place.

    Args:
        longitude: Station longitude in degrees East.
        latitude: Station latitude in degrees North.
        day_of_year: Day of year.
        ut_hour: Universal Time in hours (0-24).
        min_altitude: Minimum solar altitude for useful observation (degrees).

    Returns:
        True if the Sun is above min_altitude.
    """
    dec = solar_declination(np.array([day_of_year]))[0]
    # Hour angle: 0 at local solar noon
    local_solar_time = ut_hour + longitude / 15.0
    hour_angle = (local_solar_time - 12.0) * 15.0  # degrees

    lat_rad = np.radians(latitude)
    dec_rad = np.radians(dec)
    ha_rad = np.radians(hour_angle)

    # Solar altitude
    sin_alt = (np.sin(lat_rad) * np.sin(dec_rad) +
               np.cos(lat_rad) * np.cos(dec_rad) * np.cos(ha_rad))
    altitude = np.degrees(np.arcsin(np.clip(sin_alt, -1, 1)))
    return altitude > min_altitude


# Compute annual sunlight hours for each station
days = np.arange(1, 366)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Sunlight hours per station across the year
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, 6))
for i, station in enumerate(BISON_STATIONS):
    hours = sunlight_hours(station.latitude, days)
    ax.plot(days, hours, color=colors[i], lw=1.5, label=station.name)

ax.set_xlabel("Day of Year / 연중 일수")
ax.set_ylabel("Sunlight Hours / 일조 시간")
ax.set_title("(a) Sunlight Hours per Station / 관측소별 일조 시간")
ax.legend(fontsize=8, ncol=2)
ax.set_xlim(1, 365)
ax.set_ylim(8, 16)

# (b) Network coverage simulation for a full year
# Simulate at 1-hour resolution
ax = axes[1]
n_days = 365
hours_per_day = 24
total_hours = n_days * hours_per_day

network_coverage = np.zeros(total_hours, dtype=int)
station_coverage = np.zeros((6, total_hours), dtype=bool)

for h in range(total_hours):
    day = h // 24 + 1
    ut = h % 24
    for i, station in enumerate(BISON_STATIONS):
        if is_sun_up(station.longitude, station.latitude, day, ut, min_altitude=10.0):
            station_coverage[i, h] = True
            network_coverage[h] += 1

# Count hours with at least 1 station observing
any_station = network_coverage >= 1
theoretical_duty_cycle = np.sum(any_station) / total_hours * 100

# Monthly duty cycle
monthly_dc = []
for month_start in range(0, 365, 30):
    month_end = min(month_start + 30, 365)
    h_start = month_start * 24
    h_end = month_end * 24
    dc = np.sum(network_coverage[h_start:h_end] >= 1) / (h_end - h_start) * 100
    monthly_dc.append(dc)

months = np.arange(len(monthly_dc)) + 1
ax.bar(months, monthly_dc, color="steelblue", alpha=0.8, edgecolor="navy")
ax.axhline(theoretical_duty_cycle, color="red", ls="--", lw=2,
           label=f"Annual average: {theoretical_duty_cycle:.1f}%")
ax.axhline(80, color="orange", ls=":", lw=2, label="Paper limit: ~80%")
ax.set_xlabel("Month / 월")
ax.set_ylabel("Duty Cycle (%)")
ax.set_title("(b) Theoretical Monthly Duty Cycle / 이론적 월별 Duty Cycle")
ax.legend()
ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

print(f"Theoretical annual duty cycle (no weather/equipment): {theoretical_duty_cycle:.1f}%")
print(f"Paper reported: 78% (1994) with weather & equipment losses")

## Part 3: Window Function and Sideband Analysis / Window Function과 Sideband 분석

시계열 데이터의 간격(gap)이 파워 스펙트럼에 미치는 영향을 시각적으로 보여줍니다. 단일 관측소 vs 네트워크의 window function을 비교합니다.

We demonstrate how gaps in the time series affect the power spectrum. We compare the window function of a single site vs. the full network.

핵심 관계: 관측된 파워 스펙트럼 = 실제 스펙트럼 ⊗ window function의 Fourier 변환
Key relation: Observed power spectrum = True spectrum ⊗ Fourier transform of window function

단일 관측소의 sideband 간격: $\Delta f = 1/\text{day} = 11.6\,\mu\text{Hz}$
Single-site sideband spacing: $\Delta f = 1/\text{day} = 11.6\,\mu\text{Hz}$

In [ ]:
def create_window_function(
    n_days: int, dt_seconds: float, station_indices: list[int],
    weather_prob: float = 1.0
) -> np.ndarray:
    """Create a window function for selected BiSON stations.

    Args:
        n_days: Number of days to simulate.
        dt_seconds: Time resolution in seconds.
        station_indices: Indices into BISON_STATIONS to include.
        weather_prob: Probability of good weather at each hour (0-1).

    Returns:
        Binary window function array (1 = observing, 0 = gap).
    """
    n_points = int(n_days * 86400 / dt_seconds)
    window = np.zeros(n_points)

    for i in range(n_points):
        t_seconds = i * dt_seconds
        day = int(t_seconds / 86400) + 1
        ut_hour = (t_seconds % 86400) / 3600.0

        for idx in station_indices:
            s = BISON_STATIONS[idx]
            if is_sun_up(s.longitude, s.latitude, day, ut_hour, min_altitude=10.0):
                if weather_prob >= 1.0 or RNG.random() < weather_prob:
                    window[i] = 1.0
                    break
    return window


# Simulate 90 days at 40-second resolution (matching BiSON data cadence)
DT = 40.0  # seconds — BiSON integration time
N_DAYS_SIM = 90

# Single site (Izaña) vs full network
print("Generating window functions (this may take a moment)...")
window_single = create_window_function(N_DAYS_SIM, DT, [2])  # Izaña only
window_network = create_window_function(N_DAYS_SIM, DT, list(range(6)))

dc_single = np.mean(window_single) * 100
dc_network = np.mean(window_network) * 100
print(f"Single site (Izaña) duty cycle: {dc_single:.1f}%")
print(f"Full network duty cycle: {dc_network:.1f}%")

# Compute spectral windows (Fourier transform of window function)
def spectral_window(window: np.ndarray, dt: float) -> tuple[np.ndarray, np.ndarray]:
    """Compute the spectral window (power spectrum of the window function).

    Args:
        window: Binary window function.
        dt: Time step in seconds.

    Returns:
        Tuple of (frequencies in µHz, normalized power).
    """
    n = len(window)
    ft = np.fft.rfft(window - np.mean(window))
    power = np.abs(ft) ** 2
    power = power / np.max(power)  # Normalize
    freqs = np.fft.rfftfreq(n, d=dt) * 1e6  # Convert Hz to µHz
    return freqs, power


freqs_s, power_s = spectral_window(window_single, DT)
freqs_n, power_n = spectral_window(window_network, DT)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# (a) Window functions — 10 days excerpt
t_hours = np.arange(len(window_single)) * DT / 3600.0
excerpt = slice(0, int(10 * 86400 / DT))  # First 10 days

ax = axes[0, 0]
ax.fill_between(t_hours[excerpt] / 24, window_single[excerpt], alpha=0.5,
                color="orange", label=f"Izaña only ({dc_single:.0f}%)")
ax.set_xlabel("Day / 일수")
ax.set_ylabel("Window")
ax.set_title("(a) Single-Site Window / 단일 관측소 Window (10 days)")
ax.legend()
ax.set_ylim(-0.1, 1.3)

ax = axes[0, 1]
ax.fill_between(t_hours[excerpt] / 24, window_network[excerpt], alpha=0.5,
                color="steelblue", label=f"6-station network ({dc_network:.0f}%)")
ax.set_xlabel("Day / 일수")
ax.set_ylabel("Window")
ax.set_title("(b) Network Window / 네트워크 Window (10 days)")
ax.legend()
ax.set_ylim(-0.1, 1.3)

# (c) Spectral windows — zoom to sideband region
ax = axes[1, 0]
mask_s = freqs_s < 50
ax.plot(freqs_s[mask_s], power_s[mask_s], "orange", lw=1, alpha=0.8)
ax.axvline(11.6, color="red", ls="--", alpha=0.5, label=r"$1/\mathrm{day} = 11.6\,\mu$Hz")
ax.axvline(23.2, color="red", ls=":", alpha=0.3, label=r"$2/\mathrm{day}$")
ax.set_xlabel(r"Frequency ($\mu$Hz) / 주파수")
ax.set_ylabel("Normalized Power")
ax.set_title("(c) Spectral Window: Single Site / 단일 관측소")
ax.legend(fontsize=9)
ax.set_yscale("log")
ax.set_ylim(1e-5, 1.5)

ax = axes[1, 1]
mask_n = freqs_n < 50
ax.plot(freqs_n[mask_n], power_n[mask_n], "steelblue", lw=1, alpha=0.8)
ax.axvline(11.6, color="red", ls="--", alpha=0.5, label=r"$1/\mathrm{day} = 11.6\,\mu$Hz")
ax.axvline(23.2, color="red", ls=":", alpha=0.3, label=r"$2/\mathrm{day}$")
ax.set_xlabel(r"Frequency ($\mu$Hz) / 주파수")
ax.set_ylabel("Normalized Power")
ax.set_title("(d) Spectral Window: 6-Station Network / 6개 관측소 네트워크")
ax.legend(fontsize=9)
ax.set_yscale("log")
ax.set_ylim(1e-5, 1.5)

plt.tight_layout()
plt.show()

# Sideband suppression ratio
idx_116 = np.argmin(np.abs(freqs_s - 11.6))
print(f"\nSideband at 11.6 µHz:")
print(f"  Single site: {power_s[idx_116]:.4f} (relative to peak)")
idx_116_n = np.argmin(np.abs(freqs_n - 11.6))
print(f"  Network:     {power_n[idx_116_n]:.6f} (relative to peak)")
print(f"  Suppression factor: {power_s[idx_116] / max(power_n[idx_116_n], 1e-10):.0f}x")

## Part 4: Low-Degree p-Mode Power Spectrum / 저차수 p-모드 파워 스펙트럼

논문의 Figure 16을 재현합니다. 저차수($\ell = 0, 1, 2, 3$) p-모드를 시뮬레이션하고, 단일 관측소 vs 네트워크의 파워 스펙트럼을 비교합니다. 회전 분리(rotational splitting)도 포함합니다.

We reproduce Figure 16 from the paper. We simulate low-degree ($\ell = 0, 1, 2, 3$) p-modes and compare power spectra from a single site vs. the network. Rotational splitting is included.

대간격(large separation): $\Delta\nu \approx 135\,\mu\text{Hz}$
소간격(small separation): $\delta\nu_{02} \approx 10\,\mu\text{Hz}$
회전 분리: $\delta\nu_{\text{rot}} \approx 0.4\,\mu\text{Hz}$ (for $\ell = 1$, $m = \pm 1$ splitting)

In [ ]:
def generate_p_mode_frequencies(
    n_range: tuple[int, int] = (6, 26),
    delta_nu: float = 135.0,
    nu_0: float = 3100.0,
    small_sep_02: float = 10.0,
    rot_split: float = 0.4
) -> list[dict]:
    """Generate synthetic low-degree solar p-mode frequencies.

    Uses the asymptotic relation: nu(n,l) ≈ delta_nu * (n + l/2 + epsilon) - small_sep * l(l+1)
    with rotational splitting for m-components.

    Args:
        n_range: Range of radial orders (n_min, n_max).
        delta_nu: Large frequency separation in µHz.
        nu_0: Reference frequency for n=23, l=0 in µHz.
        small_sep_02: Small separation between l=0 and l=2 in µHz.
        rot_split: Rotational splitting in µHz per m.

    Returns:
        List of dicts with keys: n, l, m, freq_uhz, amplitude.
    """
    modes = []
    epsilon = 1.5  # Phase term

    for n in range(n_range[0], n_range[1] + 1):
        for ell in range(4):  # l = 0, 1, 2, 3
            # Asymptotic frequency
            nu_nl = delta_nu * (n + ell / 2 + epsilon)
            # Small separation correction
            if ell == 2:
                nu_nl -= small_sep_02
            elif ell == 3:
                nu_nl -= small_sep_02 * 2.5  # Approximate

            # Amplitude envelope — peaks around 3 mHz
            amp = np.exp(-0.5 * ((nu_nl - 3100) / 500) ** 2)
            # Geometric visibility: l=0 strongest, decreasing with l
            visibility = {0: 1.0, 1: 0.7, 2: 0.35, 3: 0.08}[ell]
            amp *= visibility

            # Add m-components with rotational splitting
            for m in range(-ell, ell + 1):
                # In Sun-as-a-star: only l+1 of 2l+1 components visible
                # (due to symmetry, only |m| matters)
                if ell == 0:
                    m_visible = True
                elif ell == 1:
                    m_visible = abs(m) <= 1  # m = -1, 0, +1
                else:
                    m_visible = abs(m) <= 1  # Simplified: only |m| ≤ 1 visible

                if m_visible:
                    freq = nu_nl + m * rot_split
                    modes.append({
                        "n": n, "l": ell, "m": m,
                        "freq_uhz": freq,
                        "amplitude": amp * (0.8 if m != 0 else 1.0)
                    })
    return modes


def synthesize_time_series(
    modes: list[dict], dt: float, n_points: int,
    noise_level: float = 0.05
) -> np.ndarray:
    """Synthesize a velocity time series from p-mode frequencies.

    Args:
        modes: List of mode dicts from generate_p_mode_frequencies.
        dt: Time step in seconds.
        n_points: Number of time points.
        noise_level: RMS noise level relative to strongest mode.

    Returns:
        Velocity time series array.
    """
    t = np.arange(n_points) * dt  # Time in seconds
    signal = np.zeros(n_points)

    for mode in modes:
        freq_hz = mode["freq_uhz"] * 1e-6  # Convert µHz to Hz
        amp = mode["amplitude"]
        phase = RNG.uniform(0, 2 * np.pi)
        signal += amp * np.cos(2 * np.pi * freq_hz * t + phase)

    # Add noise
    signal += RNG.normal(0, noise_level, n_points)
    return signal


# Generate modes
modes = generate_p_mode_frequencies()
print(f"Generated {len(modes)} mode components")
print(f"Frequency range: {min(m['freq_uhz'] for m in modes):.0f} – "
      f"{max(m['freq_uhz'] for m in modes):.0f} µHz")

# Synthesize time series using the window functions from Part 3
n_points = len(window_single)
signal = synthesize_time_series(modes, DT, n_points, noise_level=0.03)

# Apply windows
signal_single = signal * window_single
signal_network = signal * window_network

# Compute power spectra
def power_spectrum(signal: np.ndarray, dt: float) -> tuple[np.ndarray, np.ndarray]:
    """Compute the power spectrum of a signal.

    Args:
        signal: Input time series.
        dt: Time step in seconds.

    Returns:
        Tuple of (frequencies in µHz, power).
    """
    ft = np.fft.rfft(signal)
    power = np.abs(ft) ** 2
    freqs = np.fft.rfftfreq(len(signal), d=dt) * 1e6  # µHz
    return freqs, power


freqs, power_single_site = power_spectrum(signal_single, DT)
_, power_network_obs = power_spectrum(signal_network, DT)

# Plot — reproduce Figure 16 style
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (a) Full spectrum — single site
ax = axes[0, 0]
mask = (freqs > 1000) & (freqs < 5500)
ax.plot(freqs[mask] / 1000, power_single_site[mask], "k-", lw=0.3, alpha=0.7)
ax.set_xlabel("Frequency (mHz) / 주파수")
ax.set_ylabel("Power")
ax.set_title("(a) Single Site (Izaña) / 단일 관측소")

# (b) Full spectrum — network
ax = axes[0, 1]
ax.plot(freqs[mask] / 1000, power_network_obs[mask], "k-", lw=0.3, alpha=0.7)
ax.set_xlabel("Frequency (mHz) / 주파수")
ax.set_ylabel("Power")
ax.set_title("(b) 6-Station Network / 6개 관측소 네트워크 (cf. Fig. 16)")

# (c) Zoom: l=0,2 pair — single site (sideband confusion)
ax = axes[1, 0]
# Find n=10 modes around 1600 µHz
zoom_center = 135 * (10 + 0 / 2 + 1.5)  # n=10, l=0
zoom_range = (zoom_center - 30, zoom_center + 30)
zoom_mask = (freqs > zoom_range[0]) & (freqs < zoom_range[1])
ax.plot(freqs[zoom_mask], power_single_site[zoom_mask], "k-", lw=0.5)
ax.set_xlabel(r"Frequency ($\mu$Hz) / 주파수")
ax.set_ylabel("Power")
ax.set_title(f"(c) Zoom: n≈10 region, Single Site / 단일 관측소 확대")
# Mark expected mode positions
for mode in modes:
    if zoom_range[0] < mode["freq_uhz"] < zoom_range[1] and mode["m"] == 0:
        color = {0: "red", 1: "blue", 2: "green", 3: "purple"}[mode["l"]]
        ax.axvline(mode["freq_uhz"], color=color, alpha=0.3, ls="--", lw=0.8)

# (d) Zoom: l=1 rotational splitting — network (cf. Fig. 16 inset)
ax = axes[1, 1]
# Find l=1, n=10 mode
l1_n10_modes = [m for m in modes if m["n"] == 10 and m["l"] == 1]
if l1_n10_modes:
    center = l1_n10_modes[1]["freq_uhz"]  # m=0 component
    split_range = (center - 3, center + 3)
    split_mask = (freqs > split_range[0]) & (freqs < split_range[1])
    ax.plot(freqs[split_mask], power_network_obs[split_mask], "k-", lw=1)
    for mode in l1_n10_modes:
        label = f"m = {mode['m']}"
        ax.axvline(mode["freq_uhz"], color="red", alpha=0.5, ls="--",
                   label=label, lw=1)
    ax.set_xlabel(r"Frequency ($\mu$Hz) / 주파수")
    ax.set_ylabel("Power")
    ax.set_title(r"(d) $\ell=1, n=10$ Rotational Splitting / 회전 분리 (cf. Fig. 16 inset)")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 5: Reproducing Tables II–IV / 논문 Table II–IV 성능 데이터 재현

논문의 Table II–IV에 수록된 각 관측소의 연도별 성능 데이터를 시각화합니다.

We visualize the station-by-station performance data from the paper's Tables II–IV (1992–1994).

In [ ]:
# Data from Tables II, III, IV (pp. 14)
stations = ["Mt. Wilson", "Las Campanas", "Izaña", "Sutherland", "Carnarvon", "Narrabri"]

# [operational, good, weather] for each year
data_1992 = np.array([
    [79.7, 33.8, 42.4],
    [80.6, 59.9, 74.3],
    [95.5, 54.4, 57.0],
    [93.9, 52.6, 56.0],
    [63.7, 35.9, 56.4],
    [82.4, 39.2, 47.6],
])

data_1993 = np.array([
    [81.3, 48.8, 60.0],
    [83.3, 66.3, 79.6],
    [96.0, 63.0, 65.7],
    [91.6, 53.1, 58.0],
    [29.8, 17.6, 59.0],  # Photoelastic modulator failure!
    [70.5, 38.4, 54.5],
])

data_1994 = np.array([
    [86.4, 50.2, 58.1],
    [90.7, 71.7, 79.1],
    [96.1, 63.9, 66.5],
    [87.3, 51.5, 59.0],
    [60.6, 44.4, 73.3],
    [86.0, 53.3, 62.0],
])

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
years_data = [(data_1992, "1992"), (data_1993, "1993"), (data_1994, "1994")]

x = np.arange(len(stations))
width = 0.25

for ax, (data, year) in zip(axes, years_data):
    bars1 = ax.bar(x - width, data[:, 0], width, label="Operational",
                   color="steelblue", alpha=0.8)
    bars2 = ax.bar(x, data[:, 1], width, label="Good",
                   color="darkorange", alpha=0.8)
    bars3 = ax.bar(x + width, data[:, 2], width, label="Weather",
                   color="seagreen", alpha=0.8)

    ax.set_xlabel("Station / 관측소")
    ax.set_ylabel("Duty Cycle (%)")
    ax.set_title(f"Table {'II' if year == '1992' else 'III' if year == '1993' else 'IV'}: "
                 f"BiSON Performance {year}")
    ax.set_xticks(x)
    ax.set_xticklabels(stations, rotation=45, ha="right", fontsize=9)
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8)

    # Highlight Carnarvon 1993 failure
    if year == "1993":
        ax.annotate("PEM failure!\n장비 고장!", xy=(4, 30), fontsize=8,
                    color="red", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Network-level summary
print("Network Good Duty Cycle (paper values):")
print(f"  1993: 68%")
print(f"  1994: 78%")
print(f"  Best month (1994): 94%")
print(f"  Paper conclusion: long-term limit ≈ 80%")

## Part 6: Station Count vs. Duty Cycle / 관측소 수와 Duty Cycle 관계

관측소를 1개부터 6개까지 순차적으로 추가하면서 이론적 duty cycle이 어떻게 변하는지 시뮬레이션합니다. 또한 기상 악화와 장비 고장을 확률적으로 모델링하여, 왜 80%가 실질적 상한인지를 보여줍니다.

We simulate how the theoretical duty cycle changes as stations are added from 1 to 6. We also model weather and equipment failures probabilistically to show why 80% is the practical ceiling.

In [ ]:
# Use the hourly simulation from Part 2
# station_coverage shape: (6, total_hours)

# Station addition order (historical): Izaña, Haleakala→Mt.Wilson, Carnarvon,
# Sutherland, Las Campanas, Narrabri
# We use: Izaña(2), Mt.Wilson(0), Carnarvon(4), Sutherland(3), Las Campanas(1), Narrabri(5)
addition_order = [2, 0, 4, 3, 1, 5]
station_names_ordered = [BISON_STATIONS[i].name for i in addition_order]

# Theoretical (perfect weather, no failures)
theoretical_dc = []
for n_stations in range(1, 7):
    indices = addition_order[:n_stations]
    combined = np.any(station_coverage[indices], axis=0)
    dc = np.mean(combined) * 100
    theoretical_dc.append(dc)

# With realistic weather (average ~60% clear, from paper data)
# Average weather fractions from Table IV (1994)
weather_fractions = {
    0: 0.581,  # Mt. Wilson
    1: 0.791,  # Las Campanas
    2: 0.665,  # Izaña
    3: 0.590,  # Sutherland
    4: 0.733,  # Carnarvon
    5: 0.620,  # Narrabri
}

# Simulate with weather at hourly granularity
n_trials = 20
realistic_dc_mean = []
realistic_dc_std = []

for n_stations in range(1, 7):
    indices = addition_order[:n_stations]
    trial_dcs = []
    for trial in range(n_trials):
        combined = np.zeros(total_hours, dtype=bool)
        for idx in indices:
            # For each hour the station could observe, apply weather probability
            weather_mask = RNG.random(total_hours) < weather_fractions[idx]
            combined |= (station_coverage[idx] & weather_mask)
        trial_dcs.append(np.mean(combined) * 100)
    realistic_dc_mean.append(np.mean(trial_dcs))
    realistic_dc_std.append(np.std(trial_dcs))

# With weather + equipment failure (10% downtime per station per year)
equip_dc_mean = []
equip_dc_std = []

for n_stations in range(1, 7):
    indices = addition_order[:n_stations]
    trial_dcs = []
    for trial in range(n_trials):
        combined = np.zeros(total_hours, dtype=bool)
        for idx in indices:
            # Equipment: 90% operational (random downtime blocks)
            equip_mask = np.ones(total_hours, dtype=bool)
            # Simulate random 1-week failures
            n_failures = RNG.poisson(3)  # ~3 week-long failures per year
            for _ in range(n_failures):
                fail_start = RNG.integers(0, total_hours - 168)
                equip_mask[fail_start:fail_start + 168] = False

            weather_mask = RNG.random(total_hours) < weather_fractions[idx]
            combined |= (station_coverage[idx] & weather_mask & equip_mask)
        trial_dcs.append(np.mean(combined) * 100)
    equip_dc_mean.append(np.mean(trial_dcs))
    equip_dc_std.append(np.std(trial_dcs))

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(1, 7)
ax.plot(x, theoretical_dc, "bo-", lw=2, markersize=8, label="Theoretical (perfect) / 이론적")
ax.errorbar(x, realistic_dc_mean, yerr=realistic_dc_std, fmt="gs-", lw=2, markersize=8,
            capsize=5, label="With weather / 기상 반영")
ax.errorbar(x, equip_dc_mean, yerr=equip_dc_std, fmt="r^-", lw=2, markersize=8,
            capsize=5, label="With weather + equipment / 기상+장비 고장")

ax.axhline(80, color="orange", ls="--", lw=2, alpha=0.7, label="Paper limit: ~80%")
ax.axhline(78, color="red", ls=":", lw=1.5, alpha=0.7, label="Paper 1994 result: 78%")

ax.set_xlabel("Number of Stations / 관측소 수", fontsize=13)
ax.set_ylabel("Annual Duty Cycle (%) / 연간 Duty Cycle", fontsize=13)
ax.set_title("Station Count vs. Duty Cycle / 관측소 수 대 Duty Cycle", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n({station_names_ordered[n-1]})" for n in x],
                   fontsize=9)
ax.legend(fontsize=10)
ax.set_ylim(20, 105)

plt.tight_layout()
plt.show()

# Summary table
print("\nDuty Cycle Summary / Duty Cycle 요약:")
print(f"{'Stations':>10s} {'Theoretical':>12s} {'+ Weather':>12s} {'+ Equipment':>12s}")
print("-" * 48)
for i in range(6):
    print(f"{i+1:>10d} {theoretical_dc[i]:>11.1f}% {realistic_dc_mean[i]:>11.1f}% "
          f"{equip_dc_mean[i]:>11.1f}%")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Velocity measurement / 속도 측정 | K I 770 nm resonant scattering spectrometer, $\mathcal{R} = (I_B - I_R)/(I_B + I_R)$ | GOLF/SoHO (Na D lines), radial velocity spectrographs (HARPS, ESPRESSO) |
| Duty cycle optimization / Duty cycle 최적화 | 6-station ground network, ~78% achieved | Space-based (SoHO/GOLF: ~99%), PLATO mission |
| Sideband suppression / Sideband 억제 | Multi-site window function | Continuous space observations eliminate sidebands entirely |
| Mode identification / 모드 식별 | Low-$\ell$ from integrated light | Kepler/TESS asteroseismology (same principle for distant stars) |
| Gap-filling / 간격 채우기 | AR gap-filling, window deconvolution | Inpainting, Gaussian processes, neural network interpolation |

### 이 구현에서 다룬 논문의 핵심 아이디어 / Key Paper Ideas Covered

1. **공명 산란 분광기의 $\mathcal{R}$ 비율**이 속도에 선형 비례함을 시뮬레이션으로 검증
2. **6개 관측소의 경도 분포**가 만드는 이론적 duty cycle (~95%) vs 실제 달성치 (~78%)의 격차
3. **Window function의 sideband 구조**: 단일 관측소 $11.6\,\mu\text{Hz}$ sideband vs 네트워크의 억제 효과
4. **저차수 p-모드 파워 스펙트럼**에서 회전 분리($m = \pm 1$) 분해 (Figure 16 재현)
5. **관측소 수 증가에 따른 duty cycle 포화**: 기상+장비 고장으로 80% 상한이 결정됨

### 다음 논문과의 연결 / Connection to Next Papers

| Next Paper / 다음 논문 | Connection / 연결 |
|---|---|
| #7 Burkepile et al. (2008) — Mauna Loa K-Coronagraph | 지상 관측 네트워크에서 우주 관측으로의 전환 전, 또 다른 지상 관측 사례. 산란광 제어가 핵심. / Another ground-based case before transition to space. Stray-light control is key. |
| #8 Domingo et al. (1995) — SoHO | BiSON의 지상 한계(80% duty cycle)를 극복하는 우주 기반 일진학. GOLF 장비가 BiSON과 유사한 공명 산란 기법 사용. / Space-based helioseismology overcoming BiSON's ground limit. GOLF uses similar resonant scattering technique. |